In [90]:
import pandas as pd
from tqdm import tqdm
import numpy as np
import os

In [91]:
def id_sort(data):
    data['SortKey'] = data.iloc[:, 0].str.extract(r'(\d+)').astype(int)  # 提取數字部分，轉換為數值型
    data_sorted = data.sort_values(by='SortKey').reset_index(drop=True)  # 按數字部分排序
    data_sorted = data_sorted.drop(columns=['SortKey'])  # 刪除輔助列

    return data_sorted

In [92]:
static_data_dir = r'V:\dunwei\MACE\dataset\tabular_data\static_feature/'
dynamic_ECG_Founder_signal_data_dir = r"V:\dunwei\MACE\dataset\xgboost_result\dataset_five_fold_5_10min_longer_update\sr500_0.5_50_10s_ECG_signal_35_feature_rpeak\result_use_aucpr\best_result\all_runs_combine_avg/"
dynamic_multirocket_signal_data_dir = r"V:\dunwei\MACE\dataset\multirocket_result\multirocket\resample_0\MultiRocket_40000\sr2500_500_1000_logistic_longer_51_41_100hz\combine/"
save_data_path = r"V:\dunwei\MACE\dataset\tabular_data\static_dynamic_data/"

In [93]:
if not os.path.exists(save_data_path):
    os.makedirs(save_data_path)
    print(f'create directory: {save_data_path}')
else:
    print(f'{save_data_path} already exists')

V:\dunwei\MACE\dataset\tabular_data\static_dynamic_data/ already exists


In [94]:
static_data = pd.read_csv(static_data_dir + "MACE_preprocessed_tabular_data_5features.csv")
dynamic_ECG_Founder_signal_valid_proba = pd.read_csv(dynamic_ECG_Founder_signal_data_dir + "yhat_valid_proba_sample_level.csv")
dynamic_mr_signal_test_proba = pd.read_csv(dynamic_multirocket_signal_data_dir + "test_prob.csv")
# Rename column name
static_data = static_data.rename(columns={'Transformation_ID' : 'research_id'})
dynamic_mr_signal_test_proba = dynamic_mr_signal_test_proba.rename(columns={'file_name' : 'research_id'})
display(static_data)
display(dynamic_ECG_Founder_signal_valid_proba)
display(dynamic_mr_signal_test_proba)

,research_id,count_v_ar,count_copd,smoke_try,count_dm,male,mace
0,MACE001,0.0,0.0,1.0,0.0,1,0
1,MACE002,0.0,0.0,1.0,1.0,1,0
2,MACE003,0.0,0.0,0.0,0.0,0,0
3,MACE004,0.0,0.0,0.0,0.0,1,0
4,MACE005,0.0,0.0,1.0,0.0,1,0
...,...,...,...,...,...,...,...
519,MACE520,0.0,0.0,1.0,0.0,1,0
520,MACE521,0.0,0.0,0.0,0.0,1,0
521,MACE522,0.0,0.0,1.0,1.0,1,0
522,MACE523,0.0,0.0,0.0,1.0,1,0


,research_id,yhat_valid_proba_0_avg,yhat_valid_proba_1_avg
0,MACE001,0.878913,0.121087
1,MACE001,0.899652,0.100348
2,MACE001,0.928917,0.071083
3,MACE001,0.967878,0.032122
4,MACE001,0.940765,0.059235
...,...,...,...
198021,MACE524,0.828760,0.171240
198022,MACE524,0.914720,0.085280
198023,MACE524,0.888485,0.111515
198024,MACE524,0.874220,0.125780


,research_id,yhat_test_original
0,MACE001,0.005500
1,MACE001,0.005576
2,MACE001,0.001369
3,MACE001,0.001150
4,MACE001,0.001128
...,...,...
198021,MACE524,0.003639
198022,MACE524,0.002059
198023,MACE524,0.006535
198024,MACE524,0.004351


In [ ]:
static_feature_df = static_data.iloc[:, :-1]
label_df = static_data[['research_id', 'mace']]
dynamic_founder_feature_df = dynamic_ECG_Founder_signal_valid_proba.drop(columns='yhat_valid_proba_0_avg')
dynamic_mr_feature_df = dynamic_mr_signal_test_proba
# display(static_feature_df)
# display(dynamic_founder_feature_df)
# display(dynamic_mr_feature_df)
# display(label_df)

dynamic_df = pd.concat([dynamic_founder_feature_df, dynamic_mr_feature_df['yhat_test_original']], axis=1)
tmp_static_dynamic_df = pd.merge(static_feature_df, dynamic_df, on='research_id')
static_dynamic_df = pd.merge(tmp_static_dynamic_df, label_df, on='research_id')
static_dynamic_df = static_dynamic_df.rename(columns={'yhat_valid_proba_1_avg' : 'founder_proba_1', 'yhat_test_original' : 'askna_mr_proba_1'})
display(static_dynamic_df)
static_dynamic_df.to_csv(save_data_path + "MACE_static_dynamic_founder_askna_mr_missing_data.csv", index=False)

,research_id,count_v_ar,count_copd,smoke_try,count_dm,male,founder_proba_1,askna_mr_proba_1,mace
0,MACE001,0.0,0.0,1.0,0.0,1,0.121087,0.005500,0
1,MACE001,0.0,0.0,1.0,0.0,1,0.100348,0.005576,0
2,MACE001,0.0,0.0,1.0,0.0,1,0.071083,0.001369,0
3,MACE001,0.0,0.0,1.0,0.0,1,0.032122,0.001150,0
4,MACE001,0.0,0.0,1.0,0.0,1,0.059235,0.001128,0
...,...,...,...,...,...,...,...,...,...
198021,MACE524,0.0,0.0,0.0,0.0,0,0.171240,0.003639,0
198022,MACE524,0.0,0.0,0.0,0.0,0,0.085280,0.002059,0
198023,MACE524,0.0,0.0,0.0,0.0,0,0.111515,0.006535,0
198024,MACE524,0.0,0.0,0.0,0.0,0,0.125780,0.004351,0
